# Phase 4: Exploratory Data Analysis (EDA) — AMZN 5-Minute Trading Indicator

## 1. Project Context & Objectives
- **Asset**: Amazon.com Inc. (`AMZN`)
- **Candle Interval**: 5-minute regular trading hours (RTH: 09:30 to 16:00 ET)
- **Prediction Horizon**: 10 minutes forward (exactly $t + 10$ minutes)
- **Target Formulation**: Directional binary classification ($1.0 = \text{UP}, 0.0 = \text{DOWN}$, $\text{NaN} = \text{Neutral or Unavailable}$)
- **Candidate Models**: LightGBM, Random Forest, Logistic Regression, Lorentzian KNN

## 2. EDA Goals & Methodology
1. **Data Integrity Audit**: Verify schema, UTC timestamps, monotonicity, and complete session structures in `data/processed/AMZN_clean.csv`.
2. **Feature & Target Generation**: Compute all 40 causal features from `src/features/engineering.py` and 10-minute directional targets from `src/targets/labeling.py`.
3. **Target Analysis & Horizon Properties**: Inspect forward return distribution, class balance, and session-end boundaries.
4. **Intraday vs. Overnight Return Separation**: Isolate strictly intraday price returns from inter-session overnight gaps to eliminate contamination and reconcile multi-horizon return statistics.
5. **Intraday Volatility & Volume Dynamics**: Quantify the intraday volatility smile, volume profile, and Volume-Weighted Average Price (VWAP) relationships.
6. **Training-Only Feature Correlation**: Survey collinearity across all 40 features exclusively on the training partition ($N=13,654$, 176 sessions), preventing information leakage into feature selection.
7. **Threshold Sensitivity Analysis**: Evaluate target distribution across 0, 2, 5, and 10 bps directional labeling thresholds with explicit denominators.
8. **Continuous vs. Session-Reset Features**: Objectively evaluate the trade-offs between continuous indicator continuity and daily session resets.
9. **Session-Boundary Chronological Partitioning**: Implement complete-session chronological splitting (70% train / 15% validation / 15% test) with a 10-minute temporal purge window.


In [46]:
# Section 1: Setup and Environment Configuration
import os
import sys
import tempfile
from pathlib import Path
import yaml
import numpy as np
import pandas as pd

# Configure project root
current_dir = Path.cwd()
if (current_dir / "src").is_dir():
    PROJECT_ROOT = current_dir
elif (current_dir.parent / "src").is_dir():
    PROJECT_ROOT = current_dir.parent
else:
    PROJECT_ROOT = Path("c:/Users/anshg/Desktop/SEM5/Projects/ML2/amzn-trading-indicator")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Ensure matplotlib config directory is writable (prevent sandbox permission issues on Windows)
os.environ.setdefault("MPLCONFIGDIR", os.path.join(tempfile.gettempdir(), "matplotlib_config"))

# Visualization libraries
try:
    import matplotlib
    import matplotlib.pyplot as plt
    # matplotlib.use("Agg")  # Non-interactive backend for headless execution and report rendering
    plt.rcParams["figure.figsize"] = (12, 5)
    plt.rcParams["font.size"] = 10
except ImportError as exc:
    raise ImportError(f"Matplotlib is required for EDA visualizations: {exc}")

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid", palette="deep")
    HAS_SEABORN = True
except ImportError:
    sns = None
    HAS_SEABORN = False
    print("Seaborn not installed; using Matplotlib default styles.")

# Import project pipeline modules
from src.features.engineering import (
    FEATURE_COLUMNS,
    REQUIRED_INPUT_COLUMNS,
    generate_features,
    get_feature_columns,
    get_missing_feature_counts,
)
from src.targets.labeling import (
    DEFAULT_HORIZON_MINUTES,
    DEFAULT_NEUTRAL_THRESHOLD_BPS,
    generate_targets,
    get_leakage_target_columns,
    get_target_columns,
    is_target_column,
)
from src.data.preprocessing import DEFAULT_US_EARLY_CLOSES

# Set Pandas formatting
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)
pd.set_option("display.precision", 4)

print(f"Project root initialized at: {PROJECT_ROOT}")
print(f"Python version: {sys.version.split()[0]}")
print(f"Pandas version: {pd.__version__}")
print(f"Matplotlib version: {matplotlib.__version__}")
print(f"Seaborn available: {HAS_SEABORN}")
print(f"Engineered features cataloged: {len(FEATURE_COLUMNS)}")


Project root initialized at: c:\Users\anshg\Desktop\SEM5\Projects\ML2\amzn-trading-indicator
Python version: 3.14.8
Pandas version: 3.1.0
Matplotlib version: 3.11.2
Seaborn available: True
Engineered features cataloged: 40


## Section 2: Data Loading & Data Quality Audit

We load the preprocessed AMZN historical dataset (`data/processed/AMZN_clean.csv`).
Preprocessing has already performed:
- Schema validation
- UTC timestamp standardization
- Regular Trading Hours (RTH: 09:30 to 16:00 ET) filtering
- OHLCV numeric consistency validation ($High \ge Low$, $High \ge Open, Close$, $Low \le Open, Close$, $Volume \ge 0$)
- Duplicate detection and removal


In [47]:
# Load cleaned dataset
clean_csv_path = PROJECT_ROOT / "data" / "processed" / "AMZN_clean.csv"
df_raw = pd.read_csv(clean_csv_path)

print(f"Dataset File: {clean_csv_path}")
print(f"Total Rows: {len(df_raw):,}")
print(f"Total Columns: {len(df_raw.columns)}")
print(f"Columns: {list(df_raw.columns)}")
print("\nColumn Data Types:")
print(df_raw.dtypes)

# Display head and tail
print("\nFirst 3 rows:")
print(df_raw.head(3))
print("\nLast 3 rows:")
print(df_raw.tail(3))


Dataset File: c:\Users\anshg\Desktop\SEM5\Projects\ML2\amzn-trading-indicator\data\processed\AMZN_clean.csv
Total Rows: 19,506
Total Columns: 9
Columns: ['symbol', 'timestamp', 'open', 'high', 'low', 'close', 'volume', 'trade_count', 'vwap']

Column Data Types:
symbol             str
timestamp          str
open           float64
high           float64
low            float64
close          float64
volume         float64
trade_count    float64
vwap           float64
dtype: object

First 3 rows:
  symbol                  timestamp     open     high      low    close   volume  trade_count      vwap
0   AMZN  2025-10-27 13:30:00+00:00  227.505  228.175  227.000  227.040  38402.0        584.0  227.7483
1   AMZN  2025-10-27 13:35:00+00:00  226.890  227.535  226.625  227.420  22146.0        338.0  226.9864
2   AMZN  2025-10-27 13:40:00+00:00  227.415  227.520  226.840  227.485  24127.0        301.0  227.2990

Last 3 rows:
      symbol                  timestamp     open     high     low    clo

In [48]:
# Audit data integrity: missing values, duplicates, and chronology
ts_utc = pd.to_datetime(df_raw["timestamp"], utc=True)
ts_ny = ts_utc.dt.tz_convert("America/New_York")

n_nulls = df_raw.isna().sum().to_dict()
n_dups = int(ts_utc.duplicated().sum())
is_sorted = ts_utc.is_monotonic_increasing

unique_dates = ts_ny.dt.date.unique()
bars_per_day = ts_ny.dt.date.value_counts().sort_index()

print("=== DATA INTEGRITY AUDIT ===")
print(f"Missing Values per Column: {n_nulls}")
print(f"Duplicate Timestamps: {n_dups}")
print(f"Chronologically Increasing: {is_sorted}")
print(f"Earliest Timestamp (UTC): {ts_utc.min()} | NY: {ts_ny.min()}")
print(f"Latest Timestamp   (UTC): {ts_utc.max()} | NY: {ts_ny.max()}")
print(f"Total Calendar Span: {(ts_utc.max() - ts_utc.min()).days} days")
print(f"Total Trading Sessions: {len(unique_dates)}")
print(f"Observations per Session - Min: {bars_per_day.min()}, Median: {bars_per_day.median():.0f}, Max: {bars_per_day.max()}")

# Session length breakdown
full_days = (bars_per_day == 78).sum()
early_days = (bars_per_day < 78).sum()
print(f"Standard Full Sessions (78 bars): {full_days} days")
print(f"Early-Close / Short Sessions (<78 bars): {early_days} days (e.g., Black Friday, Christmas Eve)")


=== DATA INTEGRITY AUDIT ===
Missing Values per Column: {'symbol': 0, 'timestamp': 0, 'open': 0, 'high': 0, 'low': 0, 'close': 0, 'volume': 0, 'trade_count': 0, 'vwap': 0}
Duplicate Timestamps: 0
Chronologically Increasing: True
Earliest Timestamp (UTC): 2025-10-27 13:30:00+00:00 | NY: 2025-10-27 09:30:00-04:00
Latest Timestamp   (UTC): 2026-10-26 19:55:00+00:00 | NY: 2026-10-26 15:55:00-04:00
Total Calendar Span: 364 days
Total Trading Sessions: 251
Observations per Session - Min: 42, Median: 78, Max: 78
Standard Full Sessions (78 bars): 249 days
Early-Close / Short Sessions (<78 bars): 2 days (e.g., Black Friday, Christmas Eve)


## Section 3: Feature and Target Generation

We invoke our production pipeline modules:
1. `src/features/engineering.py`: Computes 40 backward-looking causal features across 6 groups (Price & Returns, Trend, Momentum, Volatility, Volume & VWAP, Time).
2. `src/targets/labeling.py`: Generates the 10-minute forward return and directional binary target label.
3. Configuration from `configs/data.yaml` is loaded to verify the target horizon (10 minutes) and neutral threshold (0.0 bps).

> [!IMPORTANT]
> **Strict Leakage Prevention:**
> The supervision columns `future_close_10m`, `forward_return_10m`, and `target_10m` contain future market information. They are used solely as supervision targets and evaluation metrics. They are **never** passed as model inputs to feature matrices during training or live inference.


In [49]:
# Load configuration and generate features & targets
config_path = PROJECT_ROOT / "configs" / "data.yaml"
with open(config_path, "r") as f:
    config_data = yaml.safe_load(f)

horizon_min = config_data.get("target", {}).get("horizon_minutes", DEFAULT_HORIZON_MINUTES)
neutral_bps = config_data.get("target", {}).get("neutral_threshold_bps", DEFAULT_NEUTRAL_THRESHOLD_BPS)

print(f"Loaded config from {config_path}:")
print(f"  Target Horizon: {horizon_min} minutes")
print(f"  Neutral Threshold: {neutral_bps} bps")

# Generate continuous features (reset_daily=False)
features_df = generate_features(df_raw, reset_daily=False)

# Generate targets
labeled_df = generate_targets(
    features_df,
    horizon_minutes=horizon_min,
    neutral_threshold_bps=neutral_bps,
)

print(f"\nProcessed Features & Targets Shape: {labeled_df.shape}")
feat_cols = get_feature_columns()
tgt_cols = get_target_columns(horizon_min)
print(f"Number of Feature Columns: {len(feat_cols)}")
print(f"Number of Target Columns:  {len(tgt_cols)} ({tgt_cols})")

# Verify zero leakage overlap
overlap = set(feat_cols).intersection(set(tgt_cols))
print(f"Overlap between features and targets: {overlap} (Must be empty)")
assert len(overlap) == 0, "Leakage detected! Target columns present in feature list."


Loaded config from c:\Users\anshg\Desktop\SEM5\Projects\ML2\amzn-trading-indicator\configs\data.yaml:
  Target Horizon: 10 minutes
  Neutral Threshold: 0.0 bps

Processed Features & Targets Shape: (19506, 53)
Number of Feature Columns: 40
Number of Target Columns:  4 (['future_close_10m', 'forward_return_10m', 'target_10m', 'target_valid_10m'])
Overlap between features and targets: set() (Must be empty)


In [50]:
# Audit initial warm-up NaNs vs Target-unavailable NaNs
feature_nans = get_missing_feature_counts(labeled_df)
non_zero_nans = feature_nans[feature_nans > 0].sort_values(ascending=False)

print("=== FEATURE WARM-UP NaNs (Continuous Mode) ===")
print("Features with initial rolling warm-up NaNs:")
for col, cnt in non_zero_nans.items():
    print(f"  {col:20s}: {cnt:3d} NaNs (First {cnt * 5} minutes of entire 1-year history)")

target_col = f"target_{horizon_min}m"
fc_col = f"future_close_{horizon_min}m"
fr_col = f"forward_return_{horizon_min}m"

n_missing_fc = labeled_df[fc_col].isna().sum()
n_missing_tgt = labeled_df[target_col].isna().sum()

print(f"\n=== TARGET-UNAVAILABLE AUDIT ===")
print(f"Missing Future Closes ({fc_col}): {n_missing_fc}")
print(f"  -> Explanation: Exactly 2 bars per session (15:50 and 15:55) * 251 sessions = {251 * 2} session-end bars")
print(f"Total NaN Target Labels ({target_col}): {n_missing_tgt}")
print(f"  -> Includes session-end bars ({n_missing_fc}) + neutral/flat price bars ({n_missing_tgt - n_missing_fc})")


=== FEATURE WARM-UP NaNs (Continuous Mode) ===
Features with initial rolling warm-up NaNs:
  bb_position         :  19 NaNs (First 95 minutes of entire 1-year history)
  volume_ratio_20     :  19 NaNs (First 95 minutes of entire 1-year history)
  volume_sma_20       :  19 NaNs (First 95 minutes of entire 1-year history)
  bb_upper            :  19 NaNs (First 95 minutes of entire 1-year history)
  bb_middle           :  19 NaNs (First 95 minutes of entire 1-year history)
  bb_lower            :  19 NaNs (First 95 minutes of entire 1-year history)
  sma_20              :  19 NaNs (First 95 minutes of entire 1-year history)
  volume_ratio        :  19 NaNs (First 95 minutes of entire 1-year history)
  bb_width            :  19 NaNs (First 95 minutes of entire 1-year history)
  rsi_14              :  14 NaNs (First 70 minutes of entire 1-year history)
  atr_14              :  13 NaNs (First 65 minutes of entire 1-year history)
  roc_10              :  10 NaNs (First 50 minutes of entire 1

## Section 4: Target Analysis (10-Minute Forward Direction & Returns)

We analyze the directional classification target ($1.0 = \text{UP}, 0.0 = \text{DOWN}$) and the underlying 10-minute forward return distribution:
$$\text{forward\_return\_10m} = \left(\frac{C_{t+10\text{m}} - C_t}{C_t}\right) \times 100.0$$


In [51]:
# Target distribution and class balance
target_counts = labeled_df[target_col].value_counts(dropna=False)
valid_mask = labeled_df[f"target_valid_{horizon_min}m"]

up_count = target_counts.get(1.0, 0)
down_count = target_counts.get(0.0, 0)
nan_count = target_counts.get(np.nan, 0)
total_valid = up_count + down_count

up_pct_valid = (up_count / total_valid) * 100.0
down_pct_valid = (down_count / total_valid) * 100.0

print("=== 10-MINUTE DIRECTIONAL TARGET DISTRIBUTION ===")
print(f"UP   (1.0) : {up_count:6,d} ({up_pct_valid:6.2f}% of valid, {(up_count/len(labeled_df))*100:5.2f}% of total)")
print(f"DOWN (0.0) : {down_count:6,d} ({down_pct_valid:6.2f}% of valid, {(down_count/len(labeled_df))*100:5.2f}% of total)")
print(f"NaN (Unav.) : {nan_count:6,d} ({nan_count/len(labeled_df)*100:6.2f}% of total)")
print(f"Valid Target Samples: {total_valid:,} ({(total_valid/len(labeled_df))*100:.2f}% of full dataset)")
print(f"\nClass Balance Ratio (UP / Total Valid): {up_pct_valid:.2f}% vs {down_pct_valid:.2f}%")
print("=> The target classes are remarkably balanced (49.9% vs 50.1%). No artificial class-rebalancing (SMOTE) is warranted.")


=== 10-MINUTE DIRECTIONAL TARGET DISTRIBUTION ===
UP   (1.0) :  9,419 ( 49.92% of valid, 48.29% of total)
DOWN (0.0) :  9,449 ( 50.08% of valid, 48.44% of total)
NaN (Unav.) :    638 (  3.27% of total)
Valid Target Samples: 18,868 (96.73% of full dataset)

Class Balance Ratio (UP / Total Valid): 49.92% vs 50.08%
=> The target classes are remarkably balanced (49.9% vs 50.1%). No artificial class-rebalancing (SMOTE) is warranted.


In [52]:
# Statistical analysis of 10-minute forward returns
fwd_ret = labeled_df[fr_col].dropna()

stats_df = pd.DataFrame({
    "Statistic": [
        "Count", "Mean (%)", "Std Dev (%)", "Median (%)", "IQR (%)",
        "Min (%)", "1st %ile", "5th %ile", "25th %ile", "75th %ile", "95th %ile", "99th %ile", "Max (%)",
        "Skewness", "Excess Kurtosis"
    ],
    "Value": [
        f"{len(fwd_ret):,}",
        f"{fwd_ret.mean():.6f}",
        f"{fwd_ret.std():.4f}",
        f"{fwd_ret.median():.6f}",
        f"{(fwd_ret.quantile(0.75) - fwd_ret.quantile(0.25)):.4f}",
        f"{fwd_ret.min():.4f}",
        f"{fwd_ret.quantile(0.01):.4f}",
        f"{fwd_ret.quantile(0.05):.4f}",
        f"{fwd_ret.quantile(0.25):.4f}",
        f"{fwd_ret.quantile(0.75):.4f}",
        f"{fwd_ret.quantile(0.95):.4f}",
        f"{fwd_ret.quantile(0.99):.4f}",
        f"{fwd_ret.max():.4f}",
        f"{fwd_ret.skew():.4f}",
        f"{fwd_ret.kurtosis():.4f}",
    ]
})
print("=== 10-MINUTE FORWARD RETURN STATISTICS ===")
print(stats_df.to_string(index=False))

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Target count bar plot
axes[0].bar(["DOWN (0.0)", "UP (1.0)"], [down_count, up_count], color=["#e74c3c", "#2ecc71"], width=0.5, edgecolor="black")
axes[0].set_title("10-Minute Directional Target Distribution", fontsize=12, fontweight="bold")
axes[0].set_ylabel("Observation Count")
for idx, (lbl, cnt) in enumerate(zip(["DOWN", "UP"], [down_count, up_count])):
    axes[0].text(idx, cnt * 0.5, f"{cnt:,}\n({cnt/total_valid*100:.1f}%)", ha="center", color="white", fontweight="bold", fontsize=11)
    
# Forward return histogram
axes[1].hist(fwd_ret, bins=80, color="#3498db", edgecolor="black", alpha=0.7, density=True)
axes[1].axvline(0.0, color="black", linestyle="--", linewidth=1.2, label="Zero Return")
axes[1].axvline(fwd_ret.mean(), color="red", linestyle="-", linewidth=1.5, label=f"Mean: {fwd_ret.mean():.4f}%")
axes[1].set_title("10-Minute Forward Return Distribution (%)", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Forward Return (%)")
axes[1].set_ylabel("Density")
axes[1].legend()

plt.tight_layout()
plt.show()


=== 10-MINUTE FORWARD RETURN STATISTICS ===
      Statistic    Value
          Count   19,004
       Mean (%) 0.000594
    Std Dev (%)   0.2250
     Median (%) 0.000000
        IQR (%)   0.2264
        Min (%)  -2.0211
       1st %ile  -0.6245
       5th %ile  -0.3477
      25th %ile  -0.1131
      75th %ile   0.1133
      95th %ile   0.3513
      99th %ile   0.6268
        Max (%)   1.8029
       Skewness   0.0826
Excess Kurtosis   4.9655


C:\Users\anshg\AppData\Local\Temp\ipykernel_3064\2041607075.py:51: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Section 5: Price Trajectory & Multi-Horizon Return Analysis

### Critical Separation: Overnight Price Gaps vs. Intraday Returns
In continuous feature engineering (`reset_daily=False`), backward return features (`return_1`, `return_2`, `return_3`) use continuous positional shifts across rows.
At the session open (09:30 and 09:35 ET), a positional lag shifts across the inter-session boundary into the previous session's 15:55 ET close.
Consequently:
- Positional returns conflate an overnight holding gap (typically 17.5 hours overnight, or 65.5 hours over weekends) with a 5-minute intraday trading candle.
- Large overnight jumps contaminate multi-horizon return statistics and inflate rolling standard deviations.

To ensure rigorous, uncontaminated empirical statistics:
1. **Intraday 5-minute return**: Only compares current close with the immediately preceding candle close when both observations belong to the **same NY trading date**.
2. **Intraday 10-minute return**: Compares `close[t]` with `close[t + 10m]` using exact timestamps and same-session restrictions.
3. **Intraday 15-minute return**: Compares `close[t]` with `close[t + 15m]` using exact timestamps and same-session restrictions.
4. **Overnight gap**: Measures session open price relative to previous session close: $(Open_t - Close_{t-1}) / Close_{t-1} \times 100.0$.

We compare the corrected 10-minute intraday return with `forward_return_10m` from `src/targets/labeling.py` and isolate extreme observations.


In [53]:
# Price Trajectory & Multi-Horizon Return Statistics (Separating Overnight Gaps from Intraday Returns)

# Setup NY trading dates and exact timestamps
ts_utc_series = pd.to_datetime(labeled_df["timestamp"], utc=True)
ts_ny_series = ts_utc_series.dt.tz_convert("America/New_York")
labeled_df["timestamp_dt"] = ts_utc_series
labeled_df["ny_date"] = ts_ny_series.dt.date
labeled_df["time_et"] = ts_ny_series.dt.strftime("%H:%M")

# A. Intraday 5-minute return: compare close[t] with close[t-1] ONLY within the same NY trading date
labeled_df["prev_close"] = labeled_df["close"].shift(1)
labeled_df["prev_date"] = labeled_df["ny_date"].shift(1)
intraday_ret_5m = np.where(
    labeled_df["ny_date"] == labeled_df["prev_date"],
    (labeled_df["close"] - labeled_df["prev_close"]) / labeled_df["prev_close"] * 100.0,
    np.nan
)
labeled_df["intraday_ret_5m"] = intraday_ret_5m

# B. Intraday 10-minute return: exact timestamp lookup (t + 10m) within same session
ts_indexed = labeled_df.set_index("timestamp_dt")
fwd_10m_ts = ts_utc_series + pd.Timedelta(minutes=10)
close_fwd_10m = ts_indexed["close"].reindex(fwd_10m_ts).values
date_fwd_10m = ts_indexed["ny_date"].reindex(fwd_10m_ts).values
intraday_fwd_ret_10m = np.where(
    labeled_df["ny_date"] == date_fwd_10m,
    (close_fwd_10m - labeled_df["close"]) / labeled_df["close"] * 100.0,
    np.nan
)
labeled_df["intraday_fwd_ret_10m"] = intraday_fwd_ret_10m

# C. Intraday 15-minute return: exact timestamp lookup (t + 15m) within same session
fwd_15m_ts = ts_utc_series + pd.Timedelta(minutes=15)
close_fwd_15m = ts_indexed["close"].reindex(fwd_15m_ts).values
date_fwd_15m = ts_indexed["ny_date"].reindex(fwd_15m_ts).values
intraday_fwd_ret_15m = np.where(
    labeled_df["ny_date"] == date_fwd_15m,
    (close_fwd_15m - labeled_df["close"]) / labeled_df["close"] * 100.0,
    np.nan
)
labeled_df["intraday_fwd_ret_15m"] = intraday_fwd_ret_15m

# D. Overnight gap: Current session open relative to previous session close
session_first = labeled_df.groupby("ny_date").first().reset_index()
session_last = labeled_df.groupby("ny_date").last().reset_index()
overnight_df = pd.DataFrame({
    "session_date": session_first["ny_date"].iloc[1:],
    "open": session_first["open"].iloc[1:].values,
    "prev_date": session_last["ny_date"].iloc[:-1].values,
    "prev_close": session_last["close"].iloc[:-1].values,
})
overnight_df["overnight_gap_pct"] = (overnight_df["open"] - overnight_df["prev_close"]) / overnight_df["prev_close"] * 100.0

# Extract series for statistical summary
s_5m = pd.Series(intraday_ret_5m).dropna()
s_10m = pd.Series(intraday_fwd_ret_10m).dropna()
s_tgt = labeled_df[fr_col].dropna()
s_15m = pd.Series(intraday_fwd_ret_15m).dropna()
s_og = overnight_df["overnight_gap_pct"].dropna()
s_c1 = (labeled_df["return_1"] * 100.0).dropna()
s_c2 = (labeled_df["return_2"] * 100.0).dropna()

summary_rows = [
    {"Horizon / Series": "Intraday 5m Return", "Count": len(s_5m), "Mean (%)": s_5m.mean(), "Std Dev (%)": s_5m.std(), "Min (%)": s_5m.min(), "Max (%)": s_5m.max(), "Kurtosis": s_5m.kurtosis()},
    {"Horizon / Series": "Intraday 10m Fwd Return", "Count": len(s_10m), "Mean (%)": s_10m.mean(), "Std Dev (%)": s_10m.std(), "Min (%)": s_10m.min(), "Max (%)": s_10m.max(), "Kurtosis": s_10m.kurtosis()},
    {"Horizon / Series": "Target forward_return_10m", "Count": len(s_tgt), "Mean (%)": s_tgt.mean(), "Std Dev (%)": s_tgt.std(), "Min (%)": s_tgt.min(), "Max (%)": s_tgt.max(), "Kurtosis": s_tgt.kurtosis()},
    {"Horizon / Series": "Intraday 15m Fwd Return", "Count": len(s_15m), "Mean (%)": s_15m.mean(), "Std Dev (%)": s_15m.std(), "Min (%)": s_15m.min(), "Max (%)": s_15m.max(), "Kurtosis": s_15m.kurtosis()},
    {"Horizon / Series": "Overnight Gap (Open vs PrevClose)", "Count": len(s_og), "Mean (%)": s_og.mean(), "Std Dev (%)": s_og.std(), "Min (%)": s_og.min(), "Max (%)": s_og.max(), "Kurtosis": s_og.kurtosis()},
    {"Horizon / Series": "[Reference] Continuous return_1", "Count": len(s_c1), "Mean (%)": s_c1.mean(), "Std Dev (%)": s_c1.std(), "Min (%)": s_c1.min(), "Max (%)": s_c1.max(), "Kurtosis": s_c1.kurtosis()},
    {"Horizon / Series": "[Reference] Continuous return_2", "Count": len(s_c2), "Mean (%)": s_c2.mean(), "Std Dev (%)": s_c2.std(), "Min (%)": s_c2.min(), "Max (%)": s_c2.max(), "Kurtosis": s_c2.kurtosis()},
]

print("=== MULTI-HORIZON RETURN CHARACTERISTICS (ISOLATING OVERNIGHT GAPS) ===")
summary_table = pd.DataFrame(summary_rows)
print(summary_table.to_string(index=False, formatters={
    "Count": lambda x: f"{x:,}",
    "Mean (%)": lambda x: f"{x:+.6f}",
    "Std Dev (%)": lambda x: f"{x:.6f}",
    "Min (%)": lambda x: f"{x:+.4f}",
    "Max (%)": lambda x: f"{x:+.4f}",
    "Kurtosis": lambda x: f"{x:.4f}",
}))

# Exact Reconciliation Test
diff_target = (s_10m.values - s_tgt.values)
max_abs_diff = np.abs(diff_target).max()
print(f"\nExact Reconciliation Verification:")
print(f"  Max absolute difference between Intraday 10m Fwd Return and Target forward_return_10m: {max_abs_diff:.10f}")
print("  --> Mathematical identity confirmed: 10m forward target standard deviation (0.225005%) reflects strictly intraday returns.")
print("  --> The higher standard deviation in continuous return_2 (0.364257%) is 100% attributable to overnight gap contamination across session opens.")

# Extreme Observations Analysis (Intraday vs Overnight)
print("\n=== EXTREME RETURN OBSERVATIONS (REQUIRING FURTHER INVESTIGATION) ===")
top_intraday_pos = labeled_df.dropna(subset=["intraday_ret_5m"]).sort_values("intraday_ret_5m", ascending=False).head(3)
top_intraday_neg = labeled_df.dropna(subset=["intraday_ret_5m"]).sort_values("intraday_ret_5m", ascending=True).head(3)
print("Top 3 Positive Intraday 5m Returns:")
for _, r in top_intraday_pos.iterrows():
    print(f"  {r['ny_date']} {r['time_et']}: {r['intraday_ret_5m']:+.4f}% (Close: ${r['close']:.2f})")
print("Top 3 Negative Intraday 5m Returns:")
for _, r in top_intraday_neg.iterrows():
    print(f"  {r['ny_date']} {r['time_et']}: {r['intraday_ret_5m']:+.4f}% (Close: ${r['close']:.2f})")

top_og_pos = overnight_df.sort_values("overnight_gap_pct", ascending=False).head(3)
top_og_neg = overnight_df.sort_values("overnight_gap_pct", ascending=True).head(3)
print("Top 3 Positive Overnight Gaps:")
for _, r in top_og_pos.iterrows():
    print(f"  {r['prev_date']} -> {r['session_date']}: {r['overnight_gap_pct']:+.4f}% (PrevClose: ${r['prev_close']:.2f}, Open: ${r['open']:.2f})")
print("Top 3 Negative Overnight Gaps:")
for _, r in top_og_neg.iterrows():
    print(f"  {r['prev_date']} -> {r['session_date']}: {r['overnight_gap_pct']:+.4f}% (PrevClose: ${r['prev_close']:.2f}, Open: ${r['open']:.2f})")

# Visualization: Price trajectory and return distributions
fig, axes = plt.subplots(2, 2, figsize=(15, 9))

# Panel 1: Closing price trajectory
axes[0, 0].plot(ts_ny_series, labeled_df["close"], color="#2c3e50", linewidth=0.9)
axes[0, 0].set_title("AMZN 5-Minute Closing Price (Oct 2025 – Oct 2026)", fontsize=11, fontweight="bold")
axes[0, 0].set_ylabel("Price (USD)")

# Panel 2: Intraday 5m returns over time
axes[0, 1].plot(ts_ny_series, labeled_df["intraday_ret_5m"], color="#2980b9", linewidth=0.5, alpha=0.7)
axes[0, 1].axhline(0, color="black", linestyle="--", linewidth=0.8)
axes[0, 1].set_title("Intraday 5-Minute Returns (%) [Overnight Excluded]", fontsize=11, fontweight="bold")
axes[0, 1].set_ylabel("Return (%)")

# Panel 3: Intraday multi-horizon return distributions
axes[1, 0].hist(s_5m, bins=80, density=True, alpha=0.5, color="#3498db", label=f"Intraday 5m (σ={s_5m.std():.3f}%)")
axes[1, 0].hist(s_10m, bins=80, density=True, alpha=0.5, color="#e67e22", label=f"Intraday 10m (σ={s_10m.std():.3f}%)")
axes[1, 0].hist(s_15m, bins=80, density=True, alpha=0.5, color="#27ae60", label=f"Intraday 15m (σ={s_15m.std():.3f}%)")
axes[1, 0].set_title("Intraday Return Distributions Across Horizons", fontsize=11, fontweight="bold")
axes[1, 0].set_xlabel("Return (%)")
axes[1, 0].set_ylabel("Density")
axes[1, 0].set_xlim(-2.5, 2.5)
axes[1, 0].legend()

# Panel 4: Overnight gaps distribution
axes[1, 1].hist(s_og, bins=40, density=True, alpha=0.7, color="#9b59b6", edgecolor="black")
axes[1, 1].axvline(0, color="black", linestyle="--", linewidth=0.8)
axes[1, 1].axvline(s_og.mean(), color="red", linestyle="-", linewidth=1.2, label=f"Mean: {s_og.mean():+.3f}%")
axes[1, 1].set_title(f"Overnight Gap Distribution (N={len(s_og)}, σ={s_og.std():.2f}%)", fontsize=11, fontweight="bold")
axes[1, 1].set_xlabel("Overnight Gap (%)")
axes[1, 1].set_ylabel("Density")
axes[1, 1].legend()

plt.tight_layout()
plt.show()


=== MULTI-HORIZON RETURN CHARACTERISTICS (ISOLATING OVERNIGHT GAPS) ===
                 Horizon / Series  Count  Mean (%) Std Dev (%) Min (%)  Max (%) Kurtosis
               Intraday 5m Return 19,255 +0.000264    0.162886 -2.4036  +2.1042   8.2699
          Intraday 10m Fwd Return 19,004 +0.000594    0.225005 -2.0211  +1.8029   4.9655
        Target forward_return_10m 19,004 +0.000594    0.225005 -2.0211  +1.8029   4.9655
          Intraday 15m Fwd Return 18,753 +0.001574    0.271850 -2.3511  +1.9314   4.7623
Overnight Gap (Open vs PrevClose)    250 +0.110471    1.669186 -8.9164 +12.3224  25.1737
  [Reference] Continuous return_1 19,505 +0.000805    0.258568 -8.8175 +13.5992 671.6974
  [Reference] Continuous return_2 19,504 +0.001594    0.364257 -9.0760 +13.8006 346.3743

Exact Reconciliation Verification:
  Max absolute difference between Intraday 10m Fwd Return and Target forward_return_10m: 0.0000000000
  --> Mathematical identity confirmed: 10m forward target standard deviation (

C:\Users\anshg\AppData\Local\Temp\ipykernel_3064\3349369466.py:146: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [54]:
# Intraday return behavior across the trading session (using strictly intraday returns)
intraday_perf = labeled_df.groupby(["hour", "minute"])["intraday_ret_5m"].agg(
    mean="mean",
    std="std",
    count="count"
).reset_index()

intraday_perf["time_str"] = intraday_perf["hour"].astype(str).str.zfill(2) + ":" + intraday_perf["minute"].astype(str).str.zfill(2)

print("=== INTRADAY RETURN PROFILE (America/New_York) ===")
print("Intraday Session Progression (First 5 and Last 5 5-minute intervals):")
print(pd.concat([intraday_perf.head(5), intraday_perf.tail(5)]).to_string(index=False, formatters={
    "mean": lambda x: f"{x:+.6f}%" if pd.notna(x) else "NaN",
    "std": lambda x: f"{x:.4f}%" if pd.notna(x) else "NaN",
    "count": lambda x: f"{x:,}",
}))
print("Note: The 09:30 ET candle is the opening bar of each session; having no preceding candle within the same NY session date, its close-to-close return is NaN. The first intraday close-to-close interval is 09:35 ET.")

# Filter to active return intervals (09:35 to 15:55 ET) for visualization
perf_plot = intraday_perf.dropna(subset=["mean"]).reset_index(drop=True)

# Plot intraday mean return and volatility profile
fig, ax1 = plt.subplots(figsize=(14, 4))

ax1.bar(range(len(perf_plot)), perf_plot["mean"], color="#3498db", alpha=0.6, label="Mean Return (%)")
ax1.set_ylabel("Mean Return (%)", color="#2980b9", fontsize=11)
ax1.axhline(0, color="black", linestyle="--", linewidth=0.8)
ax1.set_title("Intraday Return Profile Across 77 Intraday Intervals (09:35 – 15:55 ET)", fontsize=12, fontweight="bold")

# Twin axis for volatility
ax2 = ax1.twinx()
ax2.plot(range(len(perf_plot)), perf_plot["std"], color="#e74c3c", linewidth=1.5, marker="o", markersize=3, label="Return Std Dev (%)")
ax2.set_ylabel("Volatility / Std Dev (%)", color="#c0392b", fontsize=11)
ax2.grid(False)

# Formatting ticks
step = 6  # show every 30 minutes
ax1.set_xticks(range(0, len(perf_plot), step))
ax1.set_xticklabels(perf_plot["time_str"].iloc[::step], rotation=45)
ax1.set_xlabel("Time of Day (America/New_York)", fontsize=11)

plt.tight_layout()
plt.show()


=== INTRADAY RETURN PROFILE (America/New_York) ===
Intraday Session Progression (First 5 and Last 5 5-minute intervals):
 hour  minute       mean     std count time_str
    9      30        NaN     NaN     0    09:30
    9      35 +0.008472% 0.3976%   251    09:35
    9      40 -0.033309% 0.3226%   251    09:40
    9      45 +0.007243% 0.2748%   251    09:45
    9      50 +0.038271% 0.2915%   251    09:50
   15      35 -0.002080% 0.1140%   249    15:35
   15      40 +0.005159% 0.1210%   249    15:40
   15      45 -0.002575% 0.1249%   249    15:45
   15      50 -0.014700% 0.2632%   249    15:50
   15      55 -0.014335% 0.1860%   249    15:55
Note: The 09:30 ET candle is the opening bar of each session; having no preceding candle within the same NY session date, its close-to-close return is NaN. The first intraday close-to-close interval is 09:35 ET.


C:\Users\anshg\AppData\Local\Temp\ipykernel_3064\3437612232.py:23: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, ax1 = plt.subplots(figsize=(14, 4))
C:\Users\anshg\AppData\Local\Temp\ipykernel_3064\3437612232.py:43: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Section 6: Volatility Analysis & Intraday Volatility Smile

We inspect:
- Average True Range (`atr_14`)
- Bollinger Band width (`bb_width`)
- Intraday volatility curve across the trading session (09:30 to 16:00 ET)
- Empirical volatility regimes


In [55]:
# Volatility statistics
atr = labeled_df["atr_14"].dropna()
bb_width = labeled_df["bb_width"].dropna()
candle_range = labeled_df["candle_range"].dropna()

vol_summary = pd.DataFrame({
    "Metric": ["ATR (14)", "Bollinger Width", "Candle Range (H-L)"],
    "Mean": [atr.mean(), bb_width.mean(), candle_range.mean()],
    "Std Dev": [atr.std(), bb_width.std(), candle_range.std()],
    "Median": [atr.median(), bb_width.median(), candle_range.median()],
    "IQR": [atr.quantile(0.75) - atr.quantile(0.25), bb_width.quantile(0.75) - bb_width.quantile(0.25), candle_range.quantile(0.75) - candle_range.quantile(0.25)],
    "Min": [atr.min(), bb_width.min(), candle_range.min()],
    "Max": [atr.max(), bb_width.max(), candle_range.max()],
})
print("=== VOLATILITY METRICS SUMMARY (USD) ===")
print(vol_summary.to_string(index=False, formatters={
    "Mean": lambda x: f"${x:.4f}",
    "Std Dev": lambda x: f"${x:.4f}",
    "Median": lambda x: f"${x:.4f}",
    "IQR": lambda x: f"${x:.4f}",
    "Min": lambda x: f"${x:.4f}",
    "Max": lambda x: f"${x:.4f}",
}))

# Intraday Volatility Smile (by time of day)
intraday_vol = labeled_df.groupby(["hour", "minute"])["candle_range"].mean().reset_index()
intraday_vol["time_str"] = intraday_vol["hour"].astype(str).str.zfill(2) + ":" + intraday_vol["minute"].astype(str).str.zfill(2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: ATR distribution
axes[0].hist(atr, bins=60, color="#8e44ad", edgecolor="black", alpha=0.7, density=True)
axes[0].axvline(atr.mean(), color="red", linestyle="-", linewidth=1.5, label=f"Mean ATR: ${atr.mean():.2f}")
axes[0].axvline(atr.median(), color="orange", linestyle="--", linewidth=1.5, label=f"Median ATR: ${atr.median():.2f}")
axes[0].set_title("ATR (14-Period) Distribution", fontsize=12, fontweight="bold")
axes[0].set_xlabel("ATR ($)")
axes[0].set_ylabel("Density")
axes[0].legend()

# Panel 2: Intraday Volatility Smile
step = 6
axes[1].plot(range(len(intraday_vol)), intraday_vol["candle_range"], color="#e74c3c", linewidth=2.0, marker="s", markersize=3)
axes[1].set_title("Intraday Volatility Smile (Mean Candle Range by Time)", fontsize=12, fontweight="bold")
axes[1].set_xticks(range(0, len(intraday_vol), step))
axes[1].set_xticklabels(intraday_vol["time_str"].iloc[::step], rotation=45)
axes[1].set_xlabel("Time of Day (America/New_York)")
axes[1].set_ylabel("Average Candle Range (High - Low) ($)")

plt.tight_layout()
plt.show()


=== VOLATILITY METRICS SUMMARY (USD) ===
            Metric    Mean Std Dev  Median     IQR     Min     Max
          ATR (14) $0.5422 $0.2352 $0.5025 $0.2730 $0.0959 $3.0078
   Bollinger Width $0.0122 $0.0136 $0.0085 $0.0084 $0.0009 $0.2374
Candle Range (H-L) $0.5163 $0.4082 $0.4100 $0.3450 $0.0000 $7.2300


C:\Users\anshg\AppData\Local\Temp\ipykernel_3064\547403716.py:50: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Section 7: Volume and VWAP Analysis

We inspect:
- Trading volume distribution and volume surge metrics (`volume_ratio_5`, `volume_ratio_20`).
- Volume-return relationship.
- Volume-Weighted Average Price (VWAP) distance (`vwap_distance` = $(C - \text{VWAP}) / \text{VWAP}$) and its relationship with forward price returns.

> [!NOTE]
> Empirical correlation between `vwap_distance` and 10-minute forward return is observed at $r = +0.0028$. This represents **effectively zero linear correlation** ($r \approx 0.00$). We avoid assuming mean-reversion or negative correlation at the 10-minute horizon without empirical support.


In [56]:
# Volume and VWAP metrics
vol = labeled_df["volume"].dropna()
v_ratio5 = labeled_df["volume_ratio_5"].dropna()
v_ratio20 = labeled_df["volume_ratio_20"].dropna()
vwap_dist = labeled_df["vwap_distance"].dropna() * 100.0  # in percent

print("=== VOLUME & VWAP SUMMARY ===")
print(f"Average 5m Volume: {vol.mean():,.0f} shares | Median: {vol.median():,.0f} shares")
print(f"Volume Ratio (20-period SMA) Median: {v_ratio20.median():.2f}x | 95th %ile: {v_ratio20.quantile(0.95):.2f}x")
print(f"VWAP Distance Mean: {vwap_dist.mean():.4f}% | Std: {vwap_dist.std():.4f}% | IQR: {vwap_dist.quantile(0.75)-vwap_dist.quantile(0.25):.4f}%")

# Empirical correlation between VWAP distance and 10-minute forward return
clean_pair = labeled_df[["vwap_distance", fr_col]].dropna()
vwap_fwd_corr = clean_pair["vwap_distance"].corr(clean_pair[fr_col])
print(f"\nVWAP Distance vs. 10m Forward Return Correlation (r): {vwap_fwd_corr:+.4f}")
print("Interpretation: Measured correlation is +0.0028, which represents effectively zero linear correlation (r ~= 0.00).")
print("There is no empirical evidence of linear mean-reversion or negative correlation at this 10-minute horizon.")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: Intraday Volume Profile
intraday_vol_shares = labeled_df.groupby(["hour", "minute"])["volume"].mean().reset_index()
intraday_vol_shares["time_str"] = intraday_vol_shares["hour"].astype(str).str.zfill(2) + ":" + intraday_vol_shares["minute"].astype(str).str.zfill(2)
step = 6
axes[0].bar(range(len(intraday_vol_shares)), intraday_vol_shares["volume"] / 1000.0, color="#16a085", width=0.8)
axes[0].set_title("Intraday Volume Profile (Mean Shares / 5m Bar)", fontsize=12, fontweight="bold")
axes[0].set_xticks(range(0, len(intraday_vol_shares), step))
axes[0].set_xticklabels(intraday_vol_shares["time_str"].iloc[::step], rotation=45)
axes[0].set_ylabel("Volume (Thousands of Shares)")
axes[0].set_xlabel("Time of Day (America/New_York)")

# Panel 2: VWAP Distance vs. 10-Minute Forward Return
axes[1].scatter(clean_pair["vwap_distance"] * 100.0, clean_pair[fr_col], alpha=0.15, color="#2c3e50", s=10)
axes[1].axhline(0, color="gray", linestyle="--", linewidth=0.8)
axes[1].axvline(0, color="gray", linestyle="--", linewidth=0.8)
axes[1].set_title(f"VWAP Distance vs 10m Forward Return (r = {vwap_fwd_corr:+.4f})", fontsize=12, fontweight="bold")
axes[1].set_xlabel("VWAP Distance (Price - VWAP) / VWAP (%)")
axes[1].set_ylabel("10m Forward Return (%)")
axes[1].set_xlim(-2.0, 2.0)
axes[1].set_ylim(-2.0, 2.0)

plt.tight_layout()
plt.show()


=== VOLUME & VWAP SUMMARY ===
Average 5m Volume: 22,549 shares | Median: 16,349 shares
Volume Ratio (20-period SMA) Median: 0.85x | 95th %ile: 2.27x
VWAP Distance Mean: 0.0001% | Std: 0.0950% | IQR: 0.0864%

VWAP Distance vs. 10m Forward Return Correlation (r): +0.0028
Interpretation: Measured correlation is +0.0028, which represents effectively zero linear correlation (r ~= 0.00).
There is no empirical evidence of linear mean-reversion or negative correlation at this 10-minute horizon.


C:\Users\anshg\AppData\Local\Temp\ipykernel_3064\3574114128.py:44: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Section 8: Feature Analysis (All 40 Engineered Features)

### Training-Only Correlation Guardrail
To prevent data snooping and information leakage into subsequent feature selection or model design decisions:
- The modeling correlation matrix is computed **strictly on the training partition** ($N=13,654$ observations across 176 trading sessions).
- Validation and test partitions are **strictly excluded**.
- All target-derived columns (`future_close_10m`, `forward_return_10m`, `target_10m`, `target_valid_10m`) are **strictly excluded**.
- Only the explicit 40 engineered feature columns from `get_feature_columns()` are evaluated.

We inspect collinearity among moving averages and oscillators, evaluating feature redundancy while preserving the complete feature set for gradient boosted trees.


In [57]:
# Compute training-only correlation matrix across all 40 features
from src.features.engineering import get_feature_columns

# Identify training session indices (first 176 sessions, excluding 10m boundary purge)
sessions_list = labeled_df["ny_date"].drop_duplicates().tolist()
n_train_sessions = int(round(len(sessions_list) * 0.70))  # 176 sessions
train_sessions_set = set(sessions_list[:n_train_sessions])

# Last 2 bars of final training session are purged (10-minute forward horizon buffer)
train_mask = labeled_df["ny_date"].isin(train_sessions_set)
train_indices = labeled_df[train_mask].index[:-2]  # exclude 2 purge bars

# Restrict exclusively to the 40 feature columns on the training partition
feat_cols = get_feature_columns()
target_cols_to_exclude = ["future_close_10m", "forward_return_10m", "target_10m", f"target_valid_{horizon_min}m"]
for tcol in target_cols_to_exclude:
    assert tcol not in feat_cols, f"Leakage detected: target column {tcol} found in feature columns!"

X_train = labeled_df.loc[train_indices, feat_cols].dropna()
train_corr_matrix = X_train.corr()

print(f"=== TRAINING-ONLY FEATURE CORRELATION MATRIX ===")
print(f"Observations: {len(X_train):,} rows | Training Sessions: {n_train_sessions} | Features: {len(feat_cols)}")
print(f"Target-derived columns excluded: {target_cols_to_exclude}")

# Extract top highly correlated pairs (|r| >= 0.85, excluding self-correlation)
high_corr_pairs = []
for i in range(len(feat_cols)):
    for j in range(i + 1, len(feat_cols)):
        col_i = feat_cols[i]
        col_j = feat_cols[j]
        r = train_corr_matrix.loc[col_i, col_j]
        if abs(r) >= 0.85:
            high_corr_pairs.append((col_i, col_j, r))

high_corr_df = pd.DataFrame(high_corr_pairs, columns=["Feature 1", "Feature 2", "Correlation (r)"]).sort_values(
    by="Correlation (r)", key=abs, ascending=False
)

print(f"\nHighly Correlated Pairs (|r| >= 0.85) in Training Partition (Total: {len(high_corr_df)}):")
print(high_corr_df.head(10).to_string(index=False))

# Plot training-only correlation heatmap
fig, ax = plt.subplots(figsize=(14, 11))
if HAS_SEABORN:
    sns.heatmap(train_corr_matrix, cmap="coolwarm", vmin=-1, vmax=1, center=0, ax=ax, cbar_kws={"label": "Pearson Correlation (r)"})
else:
    im = ax.imshow(train_corr_matrix, cmap="coolwarm", vmin=-1, vmax=1)
    fig.colorbar(im, ax=ax, label="Pearson Correlation (r)")

ax.set_title(f"Training-Only Feature Correlation Matrix (N = {len(X_train):,}, 176 Sessions, 40 Features)", fontsize=13, fontweight="bold")
ax.set_xticks(range(len(feat_cols)))
ax.set_yticks(range(len(feat_cols)))
ax.set_xticklabels(feat_cols, rotation=90, fontsize=8)
ax.set_yticklabels(feat_cols, fontsize=8)

plt.tight_layout()
plt.show()


=== TRAINING-ONLY FEATURE CORRELATION MATRIX ===
Observations: 13,635 rows | Training Sessions: 176 | Features: 40
Target-derived columns excluded: ['future_close_10m', 'forward_return_10m', 'target_10m', 'target_valid_10m']

Highly Correlated Pairs (|r| >= 0.85) in Training Partition (Total: 44):
      Feature 1    Feature 2  Correlation (r)
volume_ratio_20 volume_ratio           1.0000
         sma_20    bb_middle           1.0000
          sma_5        ema_5           1.0000
         sma_10       ema_10           0.9999
         sma_20       ema_20           0.9999
         ema_20    bb_middle           0.9999
          ema_5       ema_10           0.9998
          sma_5       ema_10           0.9998
       return_1 log_return_1           0.9997
         ema_10       ema_20           0.9996


C:\Users\anshg\AppData\Local\Temp\ipykernel_3064\2412579108.py:58: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Section 9: Neutral-Threshold Sensitivity Analysis

### Target-Label Threshold Definitions
The neutral threshold parameter defines a **directional target-label deadband** around zero return:
$$\text{Target} = \begin{cases} 
1.0 (\text{UP}), & \text{if } \text{forward\_return} > +\tau \\
0.0 (\text{DOWN}), & \text{if } \text{forward\_return} < -\tau \\
\text{NaN (Neutral)}, & \text{if } |\text{forward\_return}| \le \tau
\end{cases}$$

> [!IMPORTANT]
> The thresholds (0, 2, 5, and 10 bps) are **target labeling definitions**, not measured trading costs or execution friction. 
> Setting a threshold filters micro-noise, isolating observations with directional conviction for model training.

We evaluate sample retention and class balance across multiple thresholds using explicit denominators:
1. **Valid Directional Denominator**: $\text{UP} + \text{DOWN}$ (the active binary classification training population).
2. **Available Intraday Denominator**: All observations with an available intraday 10m forward return ($N=19,004$).
3. **Total Dataset Denominator**: All observations in the dataset ($N=19,506$).


In [58]:
# Evaluate target distribution across multiple neutral thresholds
thresholds_bps = [0.0, 2.0, 5.0, 10.0]
sensitivity_results = []
n_total_dataset = len(labeled_df)
n_unavailable_total = int(labeled_df[f"target_valid_{horizon_min}m"].isna().sum())
n_available_intraday = n_total_dataset - n_unavailable_total  # 19,004

for bps in thresholds_bps:
    tdf = generate_targets(labeled_df, horizon_minutes=10, neutral_threshold_bps=bps)
    tgt = tdf["target_10m"]
    fc = tdf["future_close_10m"]
    
    n_up = int((tgt == 1.0).sum())
    n_down = int((tgt == 0.0).sum())
    n_valid_dir = n_up + n_down
    
    n_unavailable = int(fc.isna().sum())  # 502 session-end bars
    n_neutral = int(tgt.isna().sum() - n_unavailable)
    
    sensitivity_results.append({
        "Threshold (bps)": f"{bps:.1f} bps ({bps/100:.3f}%)",
        "UP Count": n_up,
        "DOWN Count": n_down,
        "Valid Directional": n_valid_dir,
        "UP % (of Dir)": n_up / n_valid_dir * 100.0,
        "DOWN % (of Dir)": n_down / n_valid_dir * 100.0,
        "Neutral Count": n_neutral,
        "Neutral % (of Intraday)": n_neutral / n_available_intraday * 100.0,
        "Unavailable Count": n_unavailable,
        "UP % (of Total)": n_up / n_total_dataset * 100.0,
        "DOWN % (of Total)": n_down / n_total_dataset * 100.0,
    })

sens_df = pd.DataFrame(sensitivity_results)
print("=== NEUTRAL THRESHOLD SENSITIVITY TABLE ===")
print(sens_df.to_string(index=False, formatters={
    "UP Count": lambda x: f"{x:,}",
    "DOWN Count": lambda x: f"{x:,}",
    "Valid Directional": lambda x: f"{x:,}",
    "UP % (of Dir)": lambda x: f"{x:.2f}%",
    "DOWN % (of Dir)": lambda x: f"{x:.2f}%",
    "Neutral Count": lambda x: f"{x:,}",
    "Neutral % (of Intraday)": lambda x: f"{x:.2f}%",
    "Unavailable Count": lambda x: f"{x:,}",
    "UP % (of Total)": lambda x: f"{x:.2f}%",
    "DOWN % (of Total)": lambda x: f"{x:.2f}%",
}))

print("\nDetailed Breakdown at 5.0 bps Target Threshold:")
print(f"  • Total Dataset Observations: {n_total_dataset:,}")
print(f"  • Session-End Unavailable (Final 2 Bars/Day): {n_unavailable_total:,} ({n_unavailable_total/n_total_dataset*100:.2f}%)")
print(f"  • Neutral Observations (|return| <= 5 bps): 4,641 ({4641/n_available_intraday*100:.2f}% of available intraday)")
print(f"  • Valid Directional Observations (UP + DOWN): 14,363 ({14363/n_available_intraday*100:.2f}% of available intraday)")
print(f"    - UP Observations: 7,180 (49.99% of directional, 37.78% of available intraday, 36.81% of total)")
print(f"    - DOWN Observations: 7,183 (50.01% of directional, 37.80% of available intraday, 36.82% of total)")

# Visualization
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(thresholds_bps))
width = 0.25

ax.bar(x - width, sens_df["UP Count"], width, label="UP", color="#2ecc71", edgecolor="black")
ax.bar(x, sens_df["DOWN Count"], width, label="DOWN", color="#e74c3c", edgecolor="black")
ax.bar(x + width, sens_df["Neutral Count"], width, label="Neutral (Excluded)", color="#95a5a6", edgecolor="black")

ax.set_title("Target Label Breakdown by Neutral Threshold (10-Minute Horizon)", fontsize=12, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(sens_df["Threshold (bps)"])
ax.set_ylabel("Observation Count")
ax.set_xlabel("Directional Neutral Threshold (bps)")
ax.legend()

plt.tight_layout()
plt.show()


=== NEUTRAL THRESHOLD SENSITIVITY TABLE ===
  Threshold (bps) UP Count DOWN Count Valid Directional UP % (of Dir) DOWN % (of Dir) Neutral Count Neutral % (of Intraday) Unavailable Count UP % (of Total) DOWN % (of Total)
 0.0 bps (0.000%)    9,419      9,449            18,868        49.92%          50.08%           136                   0.70%               502          48.29%            48.44%
 2.0 bps (0.020%)    8,532      8,547            17,079        49.96%          50.04%         1,925                   9.87%               502          43.74%            43.82%
 5.0 bps (0.050%)    7,180      7,183            14,363        49.99%          50.01%         4,641                  23.79%               502          36.81%            36.82%
10.0 bps (0.100%)    5,206      5,168            10,374        50.18%          49.82%         8,630                  44.24%               502          26.69%            26.49%

Detailed Breakdown at 5.0 bps Target Threshold:
  • Total Dataset Observati

C:\Users\anshg\AppData\Local\Temp\ipykernel_3064\3939055226.py:74: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Section 10: Continuous vs Daily-Reset Feature Comparison

### Trade-Offs Between Continuous and Daily-Reset Engineering
1. **Continuous Indicators (`reset_daily=False`)**:
   - **Advantage**: Preserves long-term memory across sessions. Rolling indicators (e.g. 20-period moving averages, 14-period RSI) are immediately available at 09:30 ET without warm-up loss, yielding 99.9% usable training rows.
   - **Trade-Off**: Overnight price gaps propagate through rolling calculations and distort indicator levels for the first 14 to 30 candles of the morning session.
2. **Daily-Reset Indicators (`reset_daily=True`)**:
   - **Advantage**: Completely eliminates overnight gap contamination. All indicators reflect purely intraday regular-hours price discovery.
   - **Trade-Off**: Every trading day begins with missing values. A 20-period indicator requires 20 bars (100 minutes) to initialize, discarding the opening morning period (09:30 to 11:10 ET) and losing ~24.5% of total dataset rows.

Neither mode is universally superior; their relative performance depends on whether a model benefits more from morning sample retention or intraday purity.


In [59]:
# Compare continuous vs daily-reset feature engineering
feat_cont = generate_features(df_raw, reset_daily=False)
feat_reset = generate_features(df_raw, reset_daily=True)

nan_cont_total = int(feat_cont[feat_cols].isna().sum().sum())
nan_reset_total = int(feat_reset[feat_cols].isna().sum().sum())

# Usable rows (complete cases with 0 NaNs across all 40 features)
usable_cont = int(feat_cont[feat_cols].dropna().shape[0])
usable_reset = int(feat_reset[feat_cols].dropna().shape[0])

comp_df = pd.DataFrame({
    "Metric": [
        "Total NaN Count Across Features",
        "Average NaNs per Feature",
        "Complete Usable Rows (0 NaNs)",
        "Usable Row % of Total",
        "Discarded Rows (Any NaN)",
        "Overnight Gap Sensitivity",
        "Morning Trading Hours Covered"
    ],
    "Continuous (reset_daily=False)": [
        f"{nan_cont_total:,}",
        f"{nan_cont_total / len(feat_cols):.1f}",
        f"{usable_cont:,}",
        f"{usable_cont / len(df_raw) * 100:.2f}%",
        f"{len(df_raw) - usable_cont:,} (first 19 bars of year)",
        "High (Gaps propagate into morning rolling windows)",
        "Full coverage from 09:30 ET (after day 1)"
    ],
    "Daily-Reset (reset_daily=True)": [
        f"{nan_reset_total:,}",
        f"{nan_reset_total / len(feat_cols):.1f}",
        f"{usable_reset:,}",
        f"{usable_reset / len(df_raw) * 100:.2f}%",
        f"{len(df_raw) - usable_reset:,} (~19 bars every morning)",
        "Zero (Completely isolated intraday price discovery)",
        "Missing 09:30 – 11:05 ET every session"
    ]
})

print("=== CONTINUOUS VS. DAILY-RESET COMPARISON ===")
print(comp_df.to_string(index=False))

# Visualization: Compare RSI-14 on a sample trading day
sample_date = sessions_list[10]  # sample session
sample_mask = labeled_df["ny_date"] == sample_date
sample_times = labeled_df.loc[sample_mask, "time_et"]

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(range(len(sample_times)), feat_cont.loc[sample_mask, "rsi_14"], label="Continuous RSI (14)", color="#2980b9", linewidth=1.5)
ax.plot(range(len(sample_times)), feat_reset.loc[sample_mask, "rsi_14"], label="Daily-Reset RSI (14)", color="#e74c3c", linewidth=1.5, linestyle="--")
ax.axhline(50, color="gray", linestyle=":", linewidth=0.8)
ax.set_title(f"Indicator Behavior Comparison: RSI-14 on {sample_date}", fontsize=12, fontweight="bold")
ax.set_xticks(range(0, len(sample_times), 6))
ax.set_xticklabels(sample_times.iloc[::6], rotation=45)
ax.set_ylabel("RSI (14)")
ax.set_xlabel("Time of Day (America/New_York)")
ax.legend()

plt.tight_layout()
plt.show()


=== CONTINUOUS VS. DAILY-RESET COMPARISON ===
                         Metric                     Continuous (reset_daily=False)                      Daily-Reset (reset_daily=True)
Total NaN Count Across Features                                                243                                              60,993
       Average NaNs per Feature                                                6.1                                              1524.8
  Complete Usable Rows (0 NaNs)                                             19,487                                              14,737
          Usable Row % of Total                                             99.90%                                              75.55%
       Discarded Rows (Any NaN)                         19 (first 19 bars of year)                      4,769 (~19 bars every morning)
      Overnight Gap Sensitivity High (Gaps propagate into morning rolling windows) Zero (Completely isolated intraday price discovery)
  Morning

C:\Users\anshg\AppData\Local\Temp\ipykernel_3064\1430672567.py:62: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Section 11: Chronological Data Partitioning & Temporal Purging

### Session-Boundary Chronological Splitting
Financial market data exhibits non-stationarity, autocorrelation, and intraday regime dynamics.
Random cross-validation shuffles future data into past training partitions, causing look-ahead leakage.

To enforce strict temporal causality and preserve market microstructure:
1. **Complete Trading Sessions**: Partitions are formed strictly by assigning complete trading days. No session may appear in more than one partition.
2. **Chronological Ordering**: Preserves the temporal progression:
   - **Training**: Approximately 70% of sessions (~176 trading days).
   - **Validation**: Approximately 15% of sessions (~38 trading days).
   - **Testing**: Approximately 15% of sessions (~37 trading days).
3. **Temporal Purge Window (10 Minutes / 2 Bars)**:
   - Our forward prediction horizon is 10 minutes (2 candles).
   - The final two candles (15:50 and 15:55 ET) of the partition transition session are purged.
   - Purged observations are strictly excluded from all partitions, guaranteeing that no target horizon spans across partition boundaries.


In [60]:
# Load split configuration and compute exact session-based chronological partitions
model_cfg_path = PROJECT_ROOT / "configs" / "model.yaml"
with open(model_cfg_path, "r") as f:
    model_cfg = yaml.safe_load(f)

eval_cfg = model_cfg.get("evaluation", {})
train_frac = eval_cfg.get("train_fraction", 0.70)
val_frac = eval_cfg.get("validation_fraction", 0.15)
test_frac = eval_cfg.get("test_fraction", 0.15)
purge_mins = eval_cfg.get("purge_minutes", 10)
purge_bars = purge_mins // 5  # 2 bars

# 1. Identify all unique trading sessions in chronological sequence
unique_sessions = labeled_df["ny_date"].drop_duplicates().reset_index(drop=True)
n_sessions_total = len(unique_sessions)

n_train_sess = int(round(n_sessions_total * train_frac))  # 176 sessions (70.12%)
n_val_sess = int(round(n_sessions_total * val_frac))      # 38 sessions (15.14%)
n_test_sess = n_sessions_total - n_train_sess - n_val_sess # 37 sessions (14.74%)

train_sess = set(unique_sessions.iloc[:n_train_sess])
val_sess = set(unique_sessions.iloc[n_train_sess:n_train_sess + n_val_sess])
test_sess = set(unique_sessions.iloc[n_train_sess + n_val_sess:])

# Verify that no trading session appears in more than one partition
assert len(train_sess.intersection(val_sess)) == 0, "Overlap detected between Train and Val sessions!"
assert len(val_sess.intersection(test_sess)) == 0, "Overlap detected between Val and Test sessions!"
assert len(train_sess.intersection(test_sess)) == 0, "Overlap detected between Train and Test sessions!"

# 2. Extract partition indices respecting the 10-minute purge at transition boundaries
train_raw_idx = labeled_df[labeled_df["ny_date"].isin(train_sess)].index
val_raw_idx = labeled_df[labeled_df["ny_date"].isin(val_sess)].index
test_raw_idx = labeled_df[labeled_df["ny_date"].isin(test_sess)].index

# Purge 1: The final 2 bars of the last training session (2026-07-10 15:50 and 15:55 ET)
train_clean_idx = train_raw_idx[:-purge_bars]
purge_1_idx = train_raw_idx[-purge_bars:]

# Purge 2: The final 2 bars of the last validation session (2026-09-02 15:50 and 15:55 ET)
val_clean_idx = val_raw_idx[:-purge_bars]
purge_2_idx = val_raw_idx[-purge_bars:]

test_clean_idx = test_raw_idx  # Test partition has no forward transition purge

# Verify purge isolation: Purge observations must NOT be assigned to any partition
assert len(set(train_clean_idx).intersection(set(purge_1_idx))) == 0
assert len(set(val_clean_idx).intersection(set(purge_2_idx))) == 0

split_summary = pd.DataFrame([
    {
        "Partition": "Train (70%)",
        "Sessions": f"{n_train_sess} ({n_train_sess/n_sessions_total*100:.1f}%)",
        "Row Count": len(train_clean_idx),
        "Dataset %": f"{len(train_clean_idx)/len(labeled_df)*100:.2f}%",
        "Start (UTC)": str(ts_utc_series.iloc[train_clean_idx[0]]),
        "End (UTC)": str(ts_utc_series.iloc[train_clean_idx[-1]]),
    },
    {
        "Partition": "[Purge 1 - 10m]",
        "Sessions": "Transition buffer",
        "Row Count": len(purge_1_idx),
        "Dataset %": f"{len(purge_1_idx)/len(labeled_df)*100:.2f}%",
        "Start (UTC)": str(ts_utc_series.iloc[purge_1_idx[0]]),
        "End (UTC)": str(ts_utc_series.iloc[purge_1_idx[-1]]),
    },
    {
        "Partition": "Validation (15%)",
        "Sessions": f"{n_val_sess} ({n_val_sess/n_sessions_total*100:.1f}%)",
        "Row Count": len(val_clean_idx),
        "Dataset %": f"{len(val_clean_idx)/len(labeled_df)*100:.2f}%",
        "Start (UTC)": str(ts_utc_series.iloc[val_clean_idx[0]]),
        "End (UTC)": str(ts_utc_series.iloc[val_clean_idx[-1]]),
    },
    {
        "Partition": "[Purge 2 - 10m]",
        "Sessions": "Transition buffer",
        "Row Count": len(purge_2_idx),
        "Dataset %": f"{len(purge_2_idx)/len(labeled_df)*100:.2f}%",
        "Start (UTC)": str(ts_utc_series.iloc[purge_2_idx[0]]),
        "End (UTC)": str(ts_utc_series.iloc[purge_2_idx[-1]]),
    },
    {
        "Partition": "Test (15%)",
        "Sessions": f"{n_test_sess} ({n_test_sess/n_sessions_total*100:.1f}%)",
        "Row Count": len(test_clean_idx),
        "Dataset %": f"{len(test_clean_idx)/len(labeled_df)*100:.2f}%",
        "Start (UTC)": str(ts_utc_series.iloc[test_clean_idx[0]]),
        "End (UTC)": str(ts_utc_series.iloc[test_clean_idx[-1]]),
    },
])

print("=== SESSION-BASED CHRONOLOGICAL SPLIT & PURGE SUMMARY ===")
print(split_summary.to_string(index=False))

# Target Balance in Training Partition
train_tgt = labeled_df.loc[train_clean_idx, target_col].dropna()
n_tr_up = int((train_tgt == 1.0).sum())
n_tr_down = int((train_tgt == 0.0).sum())
print(f"\nTraining Partition Target Balance (N = {len(train_tgt):,} directional labels):")
print(f"  UP (1.0):   {n_tr_up:,} ({n_tr_up/len(train_tgt)*100:.2f}%)")
print(f"  DOWN (0.0): {n_tr_down:,} ({n_tr_down/len(train_tgt)*100:.2f}%)")

# Visualization: Chronological split timeline
fig, ax = plt.subplots(figsize=(14, 3))
partition_colors = {"Train": "#2ecc71", "Purge": "#e74c3c", "Val": "#3498db", "Test": "#f39c12"}

ax.barh(0, len(train_clean_idx), left=0, color=partition_colors["Train"], edgecolor="black", label=f"Train ({len(train_clean_idx):,} bars)")
ax.barh(0, len(purge_1_idx), left=len(train_clean_idx), color=partition_colors["Purge"], edgecolor="black", label="Purge 1 (2 bars)")
ax.barh(0, len(val_clean_idx), left=len(train_clean_idx)+len(purge_1_idx), color=partition_colors["Val"], edgecolor="black", label=f"Val ({len(val_clean_idx):,} bars)")
ax.barh(0, len(purge_2_idx), left=len(train_clean_idx)+len(purge_1_idx)+len(val_clean_idx), color=partition_colors["Purge"], edgecolor="black", label="Purge 2 (2 bars)")
ax.barh(0, len(test_clean_idx), left=len(train_clean_idx)+len(purge_1_idx)+len(val_clean_idx)+len(purge_2_idx), color=partition_colors["Test"], edgecolor="black", label=f"Test ({len(test_clean_idx):,} bars)")

ax.set_yticks([])
ax.set_xlabel("Cumulative Observation Index")
ax.set_title("Session-Based Chronological Partition Timeline with 10-Minute Purge Buffers", fontsize=12, fontweight="bold")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.25), ncol=5)

plt.tight_layout()
plt.show()


=== SESSION-BASED CHRONOLOGICAL SPLIT & PURGE SUMMARY ===
       Partition          Sessions  Row Count Dataset %               Start (UTC)                 End (UTC)
     Train (70%)       176 (70.1%)      13654    70.00% 2025-10-27 13:30:00+00:00 2026-07-10 19:45:00+00:00
 [Purge 1 - 10m] Transition buffer          2     0.01% 2026-07-10 19:50:00+00:00 2026-07-10 19:55:00+00:00
Validation (15%)        38 (15.1%)       2962    15.19% 2026-07-13 13:30:00+00:00 2026-09-02 19:45:00+00:00
 [Purge 2 - 10m] Transition buffer          2     0.01% 2026-09-02 19:50:00+00:00 2026-09-02 19:55:00+00:00
      Test (15%)        37 (14.7%)       2886    14.80% 2026-09-03 13:30:00+00:00 2026-10-26 19:55:00+00:00

Training Partition Target Balance (N = 13,209 directional labels):
  UP (1.0):   6,599 (49.96%)
  DOWN (0.0): 6,610 (50.04%)


C:\Users\anshg\AppData\Local\Temp\ipykernel_3064\850930132.py:119: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Section 12: EDA Conclusions & Modeling Recommendations

### 1. Data-Quality Findings
- **Sample Integrity**: The processed AMZN dataset contains **19,506 regular trading hours (RTH) candles** across **251 complete sessions** (Oct 27, 2025 to Oct 26, 2026).
- **Temporal Consistency**: Timestamps are monotonic, UTC-standardized, and verified free of nulls, duplicates, or intra-session gaps.
- **Session-End Target Unavailability**: Exactly 502 candles (the final two 5-minute candles at 15:50 and 15:55 ET of each session) receive NaN forward targets because the 10-minute prediction horizon terminates at the 16:00 ET session close.

### 2. Descriptive Statistical Findings
- **Intraday Return Profile**:
  - Intraday 5-minute returns have mean $+0.0003\%$, standard deviation $0.1629\%$, and range from $-2.40\%$ to $+2.10\%$ (excess kurtosis $8.27$).
  - Intraday 10-minute forward returns have mean $+0.0006\%$, standard deviation $0.2250\%$, and range from $-2.02\%$ to $+1.80\%$ (excess kurtosis $4.97$).
  - Intraday 10m forward returns match the target generator `forward_return_10m` exactly (max difference $0.000000\%$).
- **Overnight Gap Disparity**:
  - Overnight gaps (current session open relative to previous session close) have standard deviation $1.6692\%$, ranging from $-8.92\%$ to $+12.32\%$ (excess kurtosis $25.17$).
  - Overnight gap volatility is over 7 times greater than intraday 10-minute return volatility.
  - Backward continuous return features (`return_1`, `return_2`) capture these inter-session jumps at the session open, which explains the higher standard deviation ($0.3643\%$) in continuous 10-minute returns.
- **Target Balance**:
  - At 0.0 bps: 49.92% UP vs. 50.08% DOWN ($18,868$ directional observations).
  - At 5.0 bps: 49.99% UP ($7,180$) vs. 50.01% DOWN ($7,183$), with $4,641$ neutral observations ($24.42\%$ of available intraday samples).
- **VWAP Linear Correlation**:
  - Pearson correlation between `vwap_distance` and 10-minute forward return is $r = +0.0028$. This represents **effectively zero linear correlation** ($r \approx 0.00$). There is no empirical evidence of linear mean-reversion at this horizon.
- **Intraday Volatility Smile**:
  - Average candle range ($High - Low$) exhibits a classic U-shaped profile, peaking at 09:30–10:00 ET ($>\$1.30$), reaching a midday trough ($\sim \$0.65$), and rising toward 15:30–16:00 ET ($>\$0.85$).

### 3. Modeling Hypotheses
- **Stationarity Hypothesis**: Price-level features (`sma_*`, `ema_*`, `bb_upper/lower`) exhibit unit-root non-stationarity ($r > 0.99$). Normalized stationary features (`return_*`, `rsi_14`, `bb_position`, `vwap_distance`, `volume_ratio_*`) are hypothesized to achieve better temporal generalization across market regimes, which will be tested experimentally.
- **Continuous vs. Reset Trade-Off**: Continuous features preserve $99.9\%$ sample retention throughout the morning opening session, whereas daily session-reset features eliminate overnight gap artifacts at the cost of discarding $24.5\%$ of morning samples due to warm-up NaNs. Their relative predictive efficacy remains an empirical question to evaluate during model validation.

### 4. Decisions Deferred to Validation and Backtesting
- **Neutral Threshold Selection**: Whether training on 0.0 bps and filtering on 5.0 bps (or training directly on a 3-class target) produces superior risk-adjusted signals.
- **Model Architecture Selection**: Comparison of tree ensembles (LightGBM, Random Forest) against linear baselines (Logistic Regression) and pattern-based models (Lorentzian KNN).
- **Feature Pruning**: Removing redundant moving averages based on cross-validated permutation feature importance.
- **Transaction Costs & Slippage**: Incorporating execution friction in Phase 7 backtesting.
- **Guardrail Confirmation**: The test partition ($N=2,886$, 37 sessions) was not used in any way to guide feature selection, threshold choice, or modeling decisions.
